# Hangman Dataset Splitter — 97.5% Train / 2.5% Test

**Author:** Siddhant Kumawat

Standalone notebook that does **one job only**: take the raw `words_250000_train.txt` dictionary, clean it, and split it into a **97.5% training set** and a **2.5% held‑out test set**, with no overlap between the two.

This keeps evaluation honest — if you train your BiLSTM models and compute unigram/bigram/conditional‑prior statistics only from `train_words.txt`, then test the solver on `test_words.txt`, none of the test words will have been seen during training or stats computation, so any win‑rate/accuracy number you measure actually reflects generalization rather than memorization.

**Output:** two plain‑text files, `train_words.txt` and `test_words.txt`, one word per line, saved to `/kaggle/working/`.

## 0. Setup

In [ ]:
import os
import random
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

SEED = 42
random.seed(SEED)


## 1. Load the raw dictionary

Point `DATA_PATH` at wherever your dataset is attached in this Kaggle notebook. If you're not sure of the exact path, run:

```python
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
```
and copy the printed path for `words_250000_train.txt`.

In [ ]:
DATA_PATH = "/kaggle/input/training-data/words_250000_train.txt"   # <-- update to your actual path

with open(DATA_PATH, "r") as f:
    text = f.read()

words = [line.strip() for line in text.splitlines() if line.strip()]
print(f"Raw word count: {len(words)}")


## 2. Clean the word list

Same filtering rule used throughout the main training/solver notebook: alphabetic words between 3 and 30 characters, with more than one distinct letter.

In [ ]:
def clean_word_list(word_list, min_length=3, max_length=30):
    """Filter out unwanted words (too short/long, non-alphabetic, or single-letter-repeat)."""
    cleaned_words = []
    for word in word_list:
        if (
            min_length <= len(word) <= max_length
            and word.isalpha()
            and len(set(word)) > 1
        ):
            cleaned_words.append(word)
    return cleaned_words


clean_words = clean_word_list(words)
print(f"Clean word count: {len(clean_words)}")


## 3. Split into 97.5% train / 2.5% test

A simple shuffle‑then‑slice split, seeded for reproducibility. With ~227k words this naturally preserves the overall word‑length distribution closely between the two splits (verified in Section 5), without needing explicit stratification.

In [ ]:
TEST_FRACTION = 0.025   # 2.5% held out for testing
SPLIT_SEED = 42

rng = random.Random(SPLIT_SEED)
shuffled = clean_words.copy()
rng.shuffle(shuffled)

n_test = round(len(shuffled) * TEST_FRACTION)
test_words = shuffled[:n_test]
train_words = shuffled[n_test:]

print(f"Train set: {len(train_words):,} words ({len(train_words)/len(shuffled):.2%})")
print(f"Test set : {len(test_words):,} words ({len(test_words)/len(shuffled):.2%})")


## 4. Sanity checks

In [ ]:
# no word should appear in both splits
overlap = set(train_words) & set(test_words)
assert len(overlap) == 0, f"LEAKAGE DETECTED: {len(overlap)} words appear in both splits"
print("No overlap between train and test sets -- OK")

# splits should reconstruct the full cleaned pool exactly
assert len(train_words) + len(test_words) == len(clean_words)
assert set(train_words) | set(test_words) == set(clean_words)
print("Train + test exactly reconstruct the full cleaned word pool -- OK")

# no duplicate words within a split (dictionary shouldn't have dupes, but double-check)
assert len(train_words) == len(set(train_words)), "duplicate words found in train split"
assert len(test_words) == len(set(test_words)), "duplicate words found in test split"
print("No duplicate words within either split -- OK")


## 5. Word-length distribution — train vs. test

In [ ]:
train_len_dist = Counter(len(w) for w in train_words)
test_len_dist = Counter(len(w) for w in test_words)

lengths = sorted(set(train_len_dist) | set(test_len_dist))
train_pct = [train_len_dist.get(L, 0) / len(train_words) for L in lengths]
test_pct = [test_len_dist.get(L, 0) / len(test_words) for L in lengths]

fig, ax = plt.subplots(figsize=(14, 5))
width = 0.4
x = range(len(lengths))
ax.bar([i - width/2 for i in x], train_pct, width=width, label='train', color='steelblue')
ax.bar([i + width/2 for i in x], test_pct, width=width, label='test', color='salmon')
ax.set_xticks(list(x))
ax.set_xticklabels(lengths)
ax.set_xlabel('Word length')
ax.set_ylabel('Share of split')
ax.set_title('Word-length distribution: train vs. test')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Save the split files

Written as plain text, one word per line -- the same format as the original dictionary, so they drop straight into the main training/solver notebook (just point `clean_words`/`short_words` at `train_words.txt`, and use `test_words.txt` for evaluation).

In [ ]:
os.makedirs("/kaggle/working", exist_ok=True)

with open("/kaggle/working/train_words.txt", "w") as f:
    f.write("\n".join(train_words))

with open("/kaggle/working/test_words.txt", "w") as f:
    f.write("\n".join(test_words))

print("Saved:")
print(f"  /kaggle/working/train_words.txt  ({len(train_words):,} words)")
print(f"  /kaggle/working/test_words.txt   ({len(test_words):,} words)")


## 7. Preview

In [ ]:
print("Sample train words:", train_words[:10])
print("Sample test words :", test_words[:10])
